# US Flight Delay Predictor — Full BTS Dataset (2018–2023)
### Big Data Analysis — AIE College

| Item | Detail |
|------|--------|
| Source | BTS On-Time Performance via Kaggle |
| Files used | `data/Combined_YYYY.parquet` + `data/Airlines.csv` |
| Engine | Apache Spark (PySpark) |
| Models | Linear Regression · Decision Tree · Random Forest · GBT |
| Target | Predict `ArrDelay` (arrival delay in minutes) |

---

## Section 1 — Setup & SparkSession

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, avg, count, when, isnan, isnull,
    round as spark_round, min as spark_min,
    max as spark_max, stddev, percentile_approx,
    concat_ws, lit, abs as spark_abs
)
from pyspark.sql.types import DoubleType, IntegerType

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import pandas as pd
import numpy as np
import os, glob, time, warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F8F9FA',
    'axes.grid':        True,
    'grid.alpha':       0.4,
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
})
PAL = ['#1565C0','#C62828','#2E7D32','#E65100','#6A1B9A','#00695C','#4E342E']

print('Imports OK')

Imports OK


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('US Flight Delay Predictor — Full Range')
    .master('local[*]')                          # ← use ALL available CPU cores
    .config('spark.driver.memory',          '20g')
    .config('spark.driver.maxResultSize',   '6g')
    .config('spark.sql.shuffle.partitions', str(os.cpu_count() * 2))
    .config('spark.default.parallelism',    str(os.cpu_count() * 2))
    .config('spark.sql.adaptive.enabled',              'true')
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true')
    .config('spark.sql.inMemoryColumnarStorage.compressed', 'true')
    .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')
    .config('spark.local.dir', 'C:/tmp/spark-temp')
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)
print("Parallelism:", spark.sparkContext.defaultParallelism)

Spark version: 4.1.1
Parallelism: 32


---
## Section 2 — Load Data

### Folder structure expected:
```
data/
├── Combined_2018.parquet   ← we read these  
├── Combined_2019.parquet
├── Combined_2020.parquet
├── Combined_2021.parquet
├── Combined_2022.parquet
├── Combined_2023.parquet
├── Airlines.csv            ← we join this for full airline names
└── raw/                    ← NOT used (slower, same data)
```

**Why parquet and not CSV?**  
Parquet is a columnar format — Spark only reads the columns it needs instead of the entire file. For a 10 GB dataset this makes reads **5–10× faster** than CSV.

In [3]:
# ── UPDATE THIS ───────────────────────────────────────────
DATA_FOLDER  = r"C:\Users\outis\Desktop\College\Lvl 300\2nd term\Big Data\Project\Us flight delay"                   # folder containing Combined_YYYY.parquet files
AIRLINES_CSV = os.path.join(DATA_FOLDER, 'Airlines.csv')   # Code, Description
# ─────────────────────────────────────────────────────────

# Discover parquet files (Combined_YYYY only — skip raw/)
parquet_files = sorted(glob.glob(os.path.join(DATA_FOLDER, 'Combined_*.parquet')))

if not parquet_files:
    # Fallback: try CSV if parquet not found
    print('No Combined_*.parquet found — falling back to Combined_*.csv')
    parquet_files = sorted(glob.glob(os.path.join(DATA_FOLDER, 'Combined_*.csv')))
    USE_CSV = True
else:
    USE_CSV = False

print(f'Found {len(parquet_files)} data file(s):')
total_bytes = 0
for f in parquet_files:
    sz = os.path.getsize(f)
    total_bytes += sz
    print(f'  {os.path.basename(f):<35}  {sz/1e6:>8.1f} MB')

print(f'\nTotal size : {total_bytes/1e9:.2f} GB')
print(f'Airlines   : {AIRLINES_CSV}  exists={os.path.exists(AIRLINES_CSV)}')

Found 5 data file(s):
  Combined_Flights_2018.parquet           225.8 MB
  Combined_Flights_2019.parquet           308.7 MB
  Combined_Flights_2020.parquet           183.1 MB
  Combined_Flights_2021.parquet           243.0 MB
  Combined_Flights_2022.parquet           149.6 MB

Total size : 1.11 GB
Airlines   : C:\Users\outis\Desktop\College\Lvl 300\2nd term\Big Data\Project\Us flight delay\Airlines.csv  exists=True


In [ ]:
from pyspark.sql.types import (
    StructType, StructField, IntegerType, DoubleType, StringType
)

# ── Explicit schema — skips the inferSchema scan entirely ─────
# This cuts load time from ~10 min → ~1–2 min on the same data
FLIGHT_SCHEMA = StructType([
    StructField("Year",                        IntegerType(), True),
    StructField("Quarter",                     IntegerType(), True),
    StructField("Month",                       IntegerType(), True),
    StructField("DayofMonth",                  IntegerType(), True),
    StructField("DayOfWeek",                   IntegerType(), True),
    StructField("FlightDate",                  StringType(),  True),
    StructField("Marketing_Airline_Network",   StringType(),  True),
    StructField("IATA_Code_Marketing_Airline", StringType(),  True),
    StructField("Flight_Number_Marketing_Airline", IntegerType(), True),
    StructField("Operating_Airline",           StringType(),  True),
    StructField("IATA_Code_Operating_Airline", StringType(),  True),
    StructField("Tail_Number",                 StringType(),  True),
    StructField("Flight_Number_Operating_Airline", IntegerType(), True),
    StructField("OriginAirportID",             IntegerType(), True),
    StructField("Origin",                      StringType(),  True),
    StructField("OriginCityName",              StringType(),  True),
    StructField("OriginState",                 StringType(),  True),
    StructField("DestAirportID",               IntegerType(), True),
    StructField("Dest",                        StringType(),  True),
    StructField("DestCityName",                StringType(),  True),
    StructField("DestState",                   StringType(),  True),
    StructField("CRSDepTime",                  IntegerType(), True),
    StructField("DepTime",                     DoubleType(),  True),
    StructField("DepDelay",                    DoubleType(),  True),
    StructField("DepDelayMinutes",             DoubleType(),  True),
    StructField("DepDel15",                    DoubleType(),  True),
    StructField("DepartureDelayGroups",        IntegerType(), True),
    StructField("TaxiOut",                     DoubleType(),  True),
    StructField("WheelsOff",                   DoubleType(),  True),
    StructField("WheelsOn",                    DoubleType(),  True),
    StructField("TaxiIn",                      DoubleType(),  True),
    StructField("CRSArrTime",                  IntegerType(), True),
    StructField("ArrTime",                     DoubleType(),  True),
    StructField("ArrDelay",                    DoubleType(),  True),
    StructField("ArrDelayMinutes",             DoubleType(),  True),
    StructField("ArrDel15",                    DoubleType(),  True),
    StructField("ArrivalDelayGroups",          IntegerType(), True),
    StructField("Cancelled",                   DoubleType(),  True),
    StructField("CancellationCode",            StringType(),  True),
    StructField("Diverted",                    DoubleType(),  True),
    StructField("CRSElapsedTime",              DoubleType(),  True),
    StructField("ActualElapsedTime",           DoubleType(),  True),
    StructField("AirTime",                     DoubleType(),  True),
    StructField("Flights",                     DoubleType(),  True),
    StructField("Distance",                    DoubleType(),  True),
    StructField("DistanceGroup",               IntegerType(), True),
    StructField("CarrierDelay",                DoubleType(),  True),
    StructField("WeatherDelay",                DoubleType(),  True),
    StructField("NASDelay",                    DoubleType(),  True),
    StructField("SecurityDelay",               DoubleType(),  True),
    StructField("LateAircraftDelay",           DoubleType(),  True),
    StructField("DivAirportLandings",          IntegerType(), True),
])

# ── Read CSVs with explicit schema — fast single-pass ─────────
t0 = time.time()
csv_pattern = os.path.join(DATA_FOLDER, 'Combined_*.csv')

df_raw = spark.read.csv(
    csv_pattern,
    schema=FLIGHT_SCHEMA,
    header=True,
    # mergeSchema=True  ← remove this, CSV doesn't support it
)

print(f'Read time : {time.time()-t0:.1f}s')
print(f'Columns   : {len(df_raw.columns)}')
df_raw.printSchema()

In [ ]:
# Print schema — confirms BTS column names
df_raw.printSchema()

In [ ]:
# ── Read Airlines.csv (Code → full airline name) ───────────
# Columns: Code, Description
airlines_df = (
    spark.read.csv(AIRLINES_CSV, header=True, inferSchema=True)
    .select(
        col('Code').alias('airline_code'),
        col('Description').alias('airline_name')
    )
    .dropna()
    .dropDuplicates(['airline_code'])
)

print(f'Airlines loaded: {airlines_df.count()} carriers')
airlines_df.show(8, truncate=45)

---
## 🧹 Section 3 — Data Cleaning & Preprocessing

**Exact BTS column names used** (from the dataset README):

| BTS column | What it is |
|-----------|------------|
| `IATA_Code_Marketing_Airline` | Airline code (best for cross-year analysis) |
| `Origin` / `Dest` | Airport codes |
| `ArrDelay` | **Target** — arrival delay in minutes |
| `DepDelay` | Departure delay in minutes |
| `Distance` | Flight distance in miles |
| `AirTime` | Actual air time in minutes |
| `TaxiOut` | Taxi-out time in minutes |
| `CRSDepTime` | Scheduled departure time (HHMM) |
| `Year`, `Month`, `DayOfWeek` | Time features |
| `Quarter` | Quarter (1–4) |
| `Cancelled`, `Diverted` | Filter flags |
| `CarrierDelay`, `WeatherDelay`, `NASDelay` | Delay cause breakdown |

**8 cleaning steps:**

In [ ]:
# ── Step 1: Select only the columns we need ────────────────
# Using exact BTS names from the README
KEEP = [
    # Time
    'Year', 'Quarter', 'Month', 'DayOfWeek', 'DayofMonth',
    # Airline & Route
    'IATA_Code_Marketing_Airline', 'Origin', 'Dest',
    # Target
    'ArrDelay',
    # Features
    'DepDelay', 'Distance', 'AirTime', 'TaxiOut', 'CRSDepTime',
    # Filter flags
    'Cancelled', 'Diverted',
    # Delay causes (for insight analysis)
    'CarrierDelay', 'WeatherDelay', 'NASDelay',
    'SecurityDelay', 'LateAircraftDelay',
]

# Only keep columns that actually exist in the data
available = set(df_raw.columns)
select_cols = [c for c in KEEP if c in available]
missing = [c for c in KEEP if c not in available]

if missing:
    print(f'Not found (will skip): {missing}')

df = df_raw.select(select_cols)
print(f'Selected {len(select_cols)} columns: {select_cols}')

In [ ]:
# ── Step 2: Cast to correct types ─────────────────────────
DOUBLE_COLS = ['ArrDelay','DepDelay','Distance','AirTime','TaxiOut',
               'CarrierDelay','WeatherDelay','NASDelay',
               'SecurityDelay','LateAircraftDelay']
INT_COLS    = ['Year','Quarter','Month','DayOfWeek','DayofMonth',
               'CRSDepTime']

for c in DOUBLE_COLS:
    if c in df.columns:
        df = df.withColumn(c, col(c).cast(DoubleType()))
for c in INT_COLS:
    if c in df.columns:
        df = df.withColumn(c, col(c).cast(IntegerType()))

print('Types cast')

In [ ]:
# ── Step 3: Remove cancelled & diverted ───────────────────
before = df.count()

if 'Cancelled' in df.columns:
    df = df.filter((col('Cancelled') == 0) | col('Cancelled').isNull())
    df = df.drop('Cancelled')
if 'Diverted' in df.columns:
    df = df.filter((col('Diverted') == 0) | col('Diverted').isNull())
    df = df.drop('Diverted')

after = df.count()
print(f'Removed cancelled/diverted: {before - after:,} rows  →  {after:,} remain')

In [ ]:
# ── Step 4: Remove rows where target (ArrDelay) is null ───
df = df.filter(col('ArrDelay').isNotNull())
print(f'After dropping null ArrDelay: {df.count():,}')

In [ ]:
# ── Step 5: Missing value report ──────────────────────────
total = df.count()
print(f'{'Column':<35} {'Null count':>12}  {'%':>6}  Bar')
print('─' * 65)
for c in df.columns:
    n   = df.filter(col(c).isNull()).count()
    pct = n / total * 100
    bar = '█' * int(pct / 2)
    print(f'{c:<35} {n:>12,}  {pct:>5.1f}%  {bar}')

In [ ]:
# ── Step 6: Handle missing values ─────────────────────────

# Drop rows missing the core features needed for ML
df = df.dropna(subset=['ArrDelay', 'DepDelay', 'Distance'])

# Fill AirTime & TaxiOut with their medians
for c in ['AirTime', 'TaxiOut']:
    if c in df.columns:
        med = df.approxQuantile(c, [0.5], 0.01)[0]
        df  = df.fillna({c: round(med, 1)})
        print(f'  Filled {c} nulls with median = {med:.1f}')

# Fill delay causes with 0 (no delay = no cause breakdown)
cause_cols = ['CarrierDelay','WeatherDelay','NASDelay','SecurityDelay','LateAircraftDelay']
for c in cause_cols:
    if c in df.columns:
        df = df.fillna({c: 0.0})

# Fill categorical with 'Unknown'
for c in ['IATA_Code_Marketing_Airline', 'Origin', 'Dest']:
    if c in df.columns:
        df = df.fillna({c: 'UNK'})

print(f'\nRows after null handling: {df.count():,}')

In [ ]:
# ── Step 7: Outlier removal on ArrDelay & DepDelay (IQR×3) 
def iqr_filter(df, c, factor=3.0):
    q1, q3 = df.approxQuantile(c, [0.25, 0.75], 0.005)
    iqr     = q3 - q1
    lo, hi  = q1 - factor * iqr, q3 + factor * iqr
    return df.filter((col(c) >= lo) & (col(c) <= hi)), lo, hi

before = df.count()
df, a_lo, a_hi = iqr_filter(df, 'ArrDelay')
df, d_lo, d_hi = iqr_filter(df, 'DepDelay')
df, _, _        = iqr_filter(df, 'Distance')

after = df.count()
print(f'ArrDelay  kept in [{a_lo:.0f}, {a_hi:.0f}] min')
print(f'DepDelay  kept in [{d_lo:.0f}, {d_hi:.0f}] min')
print(f'Outliers removed : {before - after:,}')

In [ ]:
# ── Step 8: Engineer DepartureHour from CRSDepTime ────────
# CRSDepTime is stored as HHMM integer (e.g. 1435 = 14:35)
if 'CRSDepTime' in df.columns:
    df = df.withColumn(
        'DepHour',
        (col('CRSDepTime') / 100).cast(IntegerType())
    ).drop('CRSDepTime')
    print('✅ DepHour engineered from CRSDepTime')

# ── Join with Airlines.csv for full names ──────────────────
if 'IATA_Code_Marketing_Airline' in df.columns:
    df = df.join(
        airlines_df,
        df['IATA_Code_Marketing_Airline'] == airlines_df['airline_code'],
        how='left'
    ).drop('airline_code')
    df = df.fillna({'airline_name': 'Unknown'})
    print('✅ Airlines joined — full names available in airline_name column')

# ── Cache clean DataFrame ──────────────────────────────────
df.cache()
TOTAL = df.count()
print(f'\nClean DataFrame cached — {TOTAL:,} rows')
print(f'   Final columns: {df.columns}')

In [ ]:
# Preview clean data
df.select(
    'Year','Month','airline_name','Origin','Dest',
    'ArrDelay','DepDelay','Distance','AirTime'
).show(8, truncate=30)

In [ ]:
# ── Year range summary ─────────────────────────────────────
year_summary = (
    df.groupBy('Year')
    .agg(
        count('*').alias('flights'),
        spark_round(avg('ArrDelay'), 2).alias('avg_arr_delay'),
        spark_round(
            count(when(col('ArrDelay') > 15, 1)) * 100.0 / count('*'), 1
        ).alias('pct_delayed')
    )
    .orderBy('Year')
    .toPandas()
)
print('=== Dataset Coverage ===')
print(year_summary.to_string(index=False))
print(f'\nYear range : {int(year_summary.Year.min())} → {int(year_summary.Year.max())}')
print(f'Total      : {year_summary.flights.sum():,} flights')

---
## Section 4 — Exploratory Data Analysis
**11 visualizations** — all saved as PNG files automatically.

In [ ]:
# Helper: safe sample → pandas  (avoids pulling full data to driver)
def to_pd(spark_df, n=40000, seed=42):
    frac = min(1.0, n / TOTAL)
    return spark_df.sample(fraction=frac, seed=seed).toPandas().dropna()

def save(name):
    plt.savefig(f'{name}.png', dpi=150, bbox_inches='tight')
    print(f'Saved → {name}.png')

print('Helpers ready')

In [ ]:
# ── VIZ 1: Delay distributions ────────────────────────────
samp = to_pd(df.select('ArrDelay','DepDelay'))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Flight Delay Distributions — Full Dataset', fontsize=15, fontweight='bold')

for ax, c, color, title in zip(
    axes,
    ['ArrDelay','DepDelay'],
    [PAL[0], PAL[1]],
    ['Arrival Delay','Departure Delay']
):
    ax.hist(samp[c], bins=80, color=color, edgecolor='white', lw=0.4, alpha=0.85)
    ax.axvline(0, color='black', lw=1.5, ls='--', label='On time')
    ax.axvline(samp[c].mean(), color='gold', lw=1.5, ls='-.',
               label=f'Mean: {samp[c].mean():.1f} min')
    ax.set_xlabel('Delay (minutes)'); ax.set_ylabel('Flights')
    ax.set_title(title); ax.legend()
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f'{x/1000:.0f}k'))

plt.tight_layout(); save('viz_01_distributions'); plt.show()

In [ ]:
# ── VIZ 2: Flight status breakdown ────────────────────────
status_pd = (
    df.withColumn('status',
        when(col('ArrDelay') < -5,  'Early')
        .when(col('ArrDelay') <= 15, 'On Time')
        .when(col('ArrDelay') <= 60, 'Delayed')
        .otherwise('Severely Delayed'))
    .groupBy('status').agg(count('*').alias('flights'))
    .toPandas()
)
order = ['Early','On Time','Delayed','Severely Delayed']
status_pd = status_pd.set_index('status').reindex(order).reset_index().dropna()
status_pd['pct'] = (status_pd['flights'] / status_pd['flights'].sum() * 100).round(1)
sc = [PAL[2], PAL[0], PAL[3], PAL[1]]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Flight Status Breakdown — All Years', fontsize=15, fontweight='bold')
wedges,_,_ = axes[0].pie(status_pd['flights'], colors=sc, autopct='%1.1f%%',
                          startangle=90, pctdistance=0.75,
                          wedgeprops={'edgecolor':'white','linewidth':2})
axes[0].legend(wedges, status_pd['status'], loc='lower center',
               bbox_to_anchor=(0.5,-0.08), fontsize=9)
axes[0].set_title('Proportion')

bars = axes[1].barh(status_pd['status'], status_pd['flights'],
                    color=sc, edgecolor='white')
for bar, row in zip(bars, status_pd.itertuples()):
    axes[1].text(bar.get_width()*1.01, bar.get_y()+bar.get_height()/2,
                 f'{row.pct}%  ({row.flights/1e6:.1f}M)', va='center', fontsize=9)
axes[1].set_xlabel('Number of Flights')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f'{x/1e6:.0f}M'))
axes[1].set_title('Count')

plt.tight_layout(); save('viz_02_status'); plt.show()

In [ ]:
# ── VIZ 3: Yearly trend ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Delay Trends Over the Years', fontsize=15, fontweight='bold')

axes[0].plot(year_summary['Year'], year_summary['avg_arr_delay'],
             'o-', color=PAL[0], lw=2.5, ms=8, label='Avg Arrival Delay')
axes[0].fill_between(year_summary['Year'], year_summary['avg_arr_delay'],
                     alpha=0.12, color=PAL[0])
axes[0].axhline(0, color='grey', lw=0.8, ls=':')
axes[0].set_xlabel('Year'); axes[0].set_ylabel('Avg Delay (min)')
axes[0].set_title('Average Arrival Delay per Year'); axes[0].legend()
axes[0].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

axes[1].bar(year_summary['Year'], year_summary['pct_delayed'],
            color=PAL[3], edgecolor='white', width=0.6)
for x, y in zip(year_summary['Year'], year_summary['pct_delayed']):
    axes[1].text(x, y+0.2, f'{y:.1f}%', ha='center', fontsize=9)
axes[1].set_xlabel('Year'); axes[1].set_ylabel('% Flights Delayed >15 min')
axes[1].set_title('Delay Rate per Year')
axes[1].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.tight_layout(); save('viz_03_yearly_trend'); plt.show()

In [ ]:
# ── VIZ 4: Monthly seasonality ────────────────────────────
month_pd = (
    df.groupBy('Month')
    .agg(spark_round(avg('ArrDelay'),2).alias('avg_arr'),
         count('*').alias('flights'))
    .orderBy('Month').toPandas().dropna()
)
MN = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
month_pd['month_name'] = month_pd['Month'].apply(
    lambda m: MN[int(m)-1] if 1 <= int(m) <= 12 else str(m))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Monthly Seasonality', fontsize=15, fontweight='bold')

worst = month_pd['avg_arr'].idxmax()
best  = month_pd['avg_arr'].idxmin()
c_m   = [PAL[1] if i==worst else (PAL[2] if i==best else PAL[0])
         for i in month_pd.index]
axes[0].bar(month_pd['month_name'], month_pd['avg_arr'], color=c_m, edgecolor='white')
axes[0].axhline(month_pd['avg_arr'].mean(), color='grey', lw=1.2, ls='--', label='Annual avg')
axes[0].set_ylabel('Avg Arrival Delay (min)')
axes[0].set_title('Avg Delay by Month  (red=worst, green=best)')
axes[0].legend()

axes[1].plot(month_pd['month_name'], month_pd['flights']/1e6,
             'o-', color=PAL[4], lw=2.5, ms=8)
axes[1].fill_between(range(len(month_pd)), month_pd['flights']/1e6,
                     alpha=0.15, color=PAL[4])
axes[1].set_ylabel('Flights (millions)'); axes[1].set_title('Flight Volume by Month')

plt.tight_layout(); save('viz_04_monthly'); plt.show()

In [ ]:
# ── VIZ 5: Airline performance (using full names!) ─────────
airline_pd = (
    df.groupBy('airline_name')
    .agg(
        spark_round(avg('ArrDelay'),  2).alias('avg_arr'),
        spark_round(avg('DepDelay'),  2).alias('avg_dep'),
        count('*').alias('flights'),
        spark_round(
            count(when(col('ArrDelay') > 15, 1)) * 100.0 / count('*'), 1
        ).alias('pct_delayed')
    )
    .filter(col('flights') >= 50000)       # only significant carriers
    .orderBy('avg_arr', ascending=False)
    .toPandas()
)

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Airline Performance — Full Name Labels', fontsize=15, fontweight='bold')

c = [PAL[1] if v >= 0 else PAL[2] for v in airline_pd['avg_arr']]
axes[0,0].barh(airline_pd['airline_name'], airline_pd['avg_arr'], color=c, edgecolor='white')
axes[0,0].axvline(0, color='black', lw=0.8)
axes[0,0].set_xlabel('Avg Arrival Delay (min)')
axes[0,0].set_title('Average Arrival Delay per Airline')

axes[0,1].barh(airline_pd['airline_name'], airline_pd['pct_delayed'],
               color=PAL[3], edgecolor='white')
axes[0,1].set_xlabel('% Flights Delayed >15 min')
axes[0,1].set_title('Delay Rate per Airline')

axes[1,0].scatter(airline_pd['flights']/1e6, airline_pd['avg_arr'],
                  s=100, color=PAL[0], alpha=0.8, edgecolors='white')
for _, row in airline_pd.iterrows():
    # Shorten name for scatter label
    short = row['airline_name'].split(' d/b/a')[0].split(',')[0][:20]
    axes[1,0].annotate(short, (row['flights']/1e6, row['avg_arr']), fontsize=7, alpha=0.8)
axes[1,0].set_xlabel('Total Flights (millions)')
axes[1,0].set_ylabel('Avg Arrival Delay (min)')
axes[1,0].set_title('Volume vs Delay')

axes[1,1].scatter(airline_pd['avg_dep'], airline_pd['avg_arr'],
                  s=100, color=PAL[4], alpha=0.8, edgecolors='white')
for _, row in airline_pd.iterrows():
    short = row['airline_name'].split(' d/b/a')[0].split(',')[0][:20]
    axes[1,1].annotate(short, (row['avg_dep'], row['avg_arr']), fontsize=7, alpha=0.8)
axes[1,1].set_xlabel('Avg Departure Delay (min)')
axes[1,1].set_ylabel('Avg Arrival Delay (min)')
axes[1,1].set_title('Dep Delay vs Arr Delay by Airline')

plt.tight_layout(); save('viz_05_airlines'); plt.show()
print('\nFull airline table:')
print(airline_pd.to_string(index=False))

In [ ]:
# ── VIZ 6: Delay causes breakdown ─────────────────────────
cause_cols_present = [c for c in
    ['CarrierDelay','WeatherDelay','NASDelay','SecurityDelay','LateAircraftDelay']
    if c in df.columns]

if cause_cols_present:
    cause_pd = (
        df.filter(col('ArrDelay') > 0)    # only delayed flights
        .agg(*[spark_round(avg(c), 2).alias(c) for c in cause_cols_present])
        .toPandas().T.reset_index()
    )
    cause_pd.columns = ['cause', 'avg_minutes']
    cause_pd = cause_pd.sort_values('avg_minutes', ascending=False)
    nice_names = {
        'CarrierDelay':      'Carrier',
        'WeatherDelay':      'Weather',
        'NASDelay':          'NAS (Air Traffic)',
        'SecurityDelay':     'Security',
        'LateAircraftDelay': 'Late Aircraft',
    }
    cause_pd['label'] = cause_pd['cause'].map(nice_names)

    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.barh(cause_pd['label'], cause_pd['avg_minutes'],
                   color=PAL[:len(cause_pd)], edgecolor='white')
    for bar, v in zip(bars, cause_pd['avg_minutes']):
        ax.text(bar.get_width()+0.2, bar.get_y()+bar.get_height()/2,
                f'{v:.1f} min', va='center', fontsize=10)
    ax.set_xlabel('Avg Delay Contribution (minutes)')
    ax.set_title('What Causes Flight Delays?  (avg minutes per delayed flight)',
                 fontweight='bold')
    plt.tight_layout(); save('viz_06_delay_causes'); plt.show()

In [ ]:
# ── VIZ 7: Day-of-week patterns ───────────────────────────
if 'DayOfWeek' in df.columns:
    dow_pd = (
        df.groupBy('DayOfWeek')
        .agg(spark_round(avg('ArrDelay'),2).alias('avg_arr'),
             count('*').alias('flights'))
        .orderBy('DayOfWeek').toPandas().dropna()
    )
    DAY_NAMES = {1:'Mon',2:'Tue',3:'Wed',4:'Thu',5:'Fri',6:'Sat',7:'Sun'}
    dow_pd['day'] = dow_pd['DayOfWeek'].map(DAY_NAMES)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle('Day-of-Week Patterns', fontsize=15, fontweight='bold')

    worst_dow = dow_pd['avg_arr'].idxmax()
    c_dow = [PAL[1] if i==worst_dow else PAL[0] for i in dow_pd.index]
    axes[0].bar(dow_pd['day'], dow_pd['avg_arr'], color=c_dow, edgecolor='white')
    axes[0].set_ylabel('Avg Arrival Delay (min)')
    axes[0].set_title('Worst Day to Fly?')

    axes[1].bar(dow_pd['day'], dow_pd['flights']/1e6, color=PAL[2], edgecolor='white')
    axes[1].set_ylabel('Flights (millions)')
    axes[1].set_title('Busiest Day of Week')

    plt.tight_layout(); save('viz_07_day_of_week'); plt.show()

In [ ]:
# ── VIZ 8: Top routes ─────────────────────────────────────
if 'Origin' in df.columns and 'Dest' in df.columns:
    routes = (
        df.withColumn('route', concat_ws(' → ', 'Origin', 'Dest'))
        .groupBy('route')
        .agg(spark_round(avg('ArrDelay'),1).alias('avg_delay'),
             count('*').alias('flights'))
        .filter(col('flights') >= 10000)
    )
    top_delayed  = routes.orderBy('avg_delay',  ascending=False).limit(15).toPandas()
    most_flights = routes.orderBy('flights',    ascending=False).limit(15).toPandas()

    fig, axes = plt.subplots(1, 2, figsize=(17, 7))
    fig.suptitle('Route Analysis', fontsize=15, fontweight='bold')

    axes[0].barh(top_delayed['route'][::-1], top_delayed['avg_delay'][::-1],
                 color=PAL[1], edgecolor='white')
    axes[0].set_xlabel('Avg Arrival Delay (min)')
    axes[0].set_title('Top 15 Most Delayed Routes')

    axes[1].barh(most_flights['route'][::-1], most_flights['flights'][::-1]/1e3,
                 color=PAL[0], edgecolor='white')
    axes[1].set_xlabel('Flights (thousands)')
    axes[1].set_title('Top 15 Busiest Routes')

    plt.tight_layout(); save('viz_08_routes'); plt.show()

In [ ]:
# ── VIZ 9: Departure hour heatmap ─────────────────────────
if 'DepHour' in df.columns and 'DayOfWeek' in df.columns:
    heat_pd = (
        df.groupBy('DayOfWeek','DepHour')
        .agg(spark_round(avg('ArrDelay'),2).alias('avg_delay'))
        .toPandas().dropna()
    )
    DAY_NAMES = {1:'Mon',2:'Tue',3:'Wed',4:'Thu',5:'Fri',6:'Sat',7:'Sun'}
    heat_pd['day'] = heat_pd['DayOfWeek'].map(DAY_NAMES)
    pivot = heat_pd.pivot(index='day', columns='DepHour', values='avg_delay')
    pivot = pivot.reindex(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'])

    fig, ax = plt.subplots(figsize=(16, 5))
    sns.heatmap(pivot, cmap='RdYlGn_r', center=0, ax=ax,
                linewidths=0.3, cbar_kws={'label':'Avg Arr Delay (min)'})
    ax.set_xlabel('Hour of Day (0–23)')
    ax.set_ylabel('Day of Week')
    ax.set_title('Avg Arrival Delay by Hour & Day  (red = worst times to fly)',
                 fontweight='bold')
    plt.tight_layout(); save('viz_09_heatmap_hour_day'); plt.show()

In [ ]:
# ── VIZ 10: Correlation matrix ────────────────────────────
num_corr = [c for c in
    ['ArrDelay','DepDelay','Distance','AirTime','TaxiOut','DepHour','Month','DayOfWeek']
    if c in df.columns]

corr_pd = to_pd(df.select(num_corr), n=30000).dropna()
corr_mat = corr_pd.corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_mat, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, ax=ax, linewidths=0.5,
            cbar_kws={'shrink':0.8}, annot_kws={'size':9})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout(); save('viz_10_correlation'); plt.show()

print('\n💡 Correlation with ArrDelay:')
for c in [x for x in num_corr if x != 'ArrDelay']:
    print(f'  {c:<18} {corr_mat.loc["ArrDelay",c]:+.3f}')

In [ ]:
# ── VIZ 11: DepDelay vs ArrDelay scatter + hexbin ─────────
scat = to_pd(df.select('DepDelay','ArrDelay','Distance'), n=25000)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Key Relationships', fontsize=15, fontweight='bold')

sc = axes[0].scatter(scat['DepDelay'], scat['ArrDelay'],
                     c=scat['Distance'], cmap='viridis', alpha=0.25, s=6)
plt.colorbar(sc, ax=axes[0], label='Distance (miles)')
axes[0].set_xlabel('Dep Delay (min)'); axes[0].set_ylabel('Arr Delay (min)')
axes[0].set_title('DepDelay vs ArrDelay  (coloured by distance)')

axes[1].hexbin(scat['Distance'], scat['ArrDelay'],
               gridsize=40, cmap='YlOrRd', mincnt=1)
axes[1].set_xlabel('Distance (miles)'); axes[1].set_ylabel('Arr Delay (min)')
axes[1].set_title('Distance vs ArrDelay  (hexbin density)')

plt.tight_layout(); save('viz_11_scatter'); plt.show()

---
## Section 5 — Feature Engineering & ML Preparation

In [ ]:
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml import Pipeline

stages       = []
cat_features = []

# Encode categorical columns → numeric index
for cat_col in ['IATA_Code_Marketing_Airline', 'Origin', 'Dest']:
    if cat_col in df.columns:
        idx = StringIndexer(
            inputCol=cat_col, outputCol=f'{cat_col}_idx',
            handleInvalid='keep'
        )
        stages.append(idx)
        cat_features.append(f'{cat_col}_idx')

# Numeric features
num_features = ['DepDelay', 'Distance']
num_features += [c for c in ['AirTime','TaxiOut','DepHour','Month','DayOfWeek','Quarter']
                 if c in df.columns]

FEATURE_COLS = num_features + cat_features
LABEL_COL    = 'ArrDelay'

assembler = VectorAssembler(
    inputCols=FEATURE_COLS,
    outputCol='features',
    handleInvalid='skip'
)
stages.append(assembler)

print(f'Features ({len(FEATURE_COLS)}): {FEATURE_COLS}')
print(f'Label: {LABEL_COL}')

In [ ]:
# Build prep pipeline — encodes categoricals + assembles vector
prep_model = Pipeline(stages=stages).fit(df)
df_ml      = prep_model.transform(df).select('features', LABEL_COL)

# 80 / 20 split
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)
train_df.cache()
test_df.cache()

print(f'Train rows : {train_df.count():,}')
print(f'Test rows  : {test_df.count():,}')
df_ml.show(3, truncate=60)

---
## Section 6 — Training 4 ML Models

| # | Model | Why |
|---|-------|-----|
| 1 | **Linear Regression** | Fast baseline — assumes linear relationship between features and delay |
| 2 | **Decision Tree** | Splits data by thresholds into rules — easy to interpret and explain |
| 3 | **Random Forest** | Builds 50 trees in parallel and averages — robust, handles nonlinear patterns |
| 4 | **Gradient Boosted Trees** | Builds trees sequentially each fixing the last — usually the most accurate |

In [ ]:
from pyspark.ml.regression import (
    LinearRegression, DecisionTreeRegressor,
    RandomForestRegressor, GBTRegressor
)
from pyspark.ml.evaluation import RegressionEvaluator

def evaluate(preds):
    out = {}
    for m in ['rmse','mae','r2']:
        out[m] = RegressionEvaluator(
            labelCol=LABEL_COL, predictionCol='prediction', metricName=m
        ).evaluate(preds)
    return out

results = {}
print('Evaluators ready')

In [ ]:
# ── Model 1: Linear Regression ────────────────────────────
print('[1/4] Training Linear Regression...')
t0 = time.time()
lr_model = LinearRegression(
    featuresCol='features', labelCol=LABEL_COL,
    predictionCol='prediction',
    maxIter=100, regParam=0.1
).fit(train_df)
lr_preds = lr_model.transform(test_df)
results['Linear Regression'] = {**evaluate(lr_preds),
    'time': round(time.time()-t0,1), 'preds': lr_preds}
r = results['Linear Regression']
print(f'RMSE={r["rmse"]:.3f}  MAE={r["mae"]:.3f}  R²={r["r2"]:.4f}  ({r["time"]}s)')

In [ ]:
# ── Model 2: Decision Tree ─────────────────────────────────
print('[2/4] Training Decision Tree...')
t0 = time.time()
dt_model = DecisionTreeRegressor(
    featuresCol='features', labelCol=LABEL_COL,
    predictionCol='prediction',
    maxDepth=10, minInstancesPerNode=20
).fit(train_df)
dt_preds = dt_model.transform(test_df)
results['Decision Tree'] = {**evaluate(dt_preds),
    'time': round(time.time()-t0,1), 'preds': dt_preds,
    'importances': dt_model.featureImportances}
r = results['Decision Tree']
print(f'RMSE={r["rmse"]:.3f}  MAE={r["mae"]:.3f}  R²={r["r2"]:.4f}  ({r["time"]}s)')

In [ ]:
# ── Model 3: Random Forest ─────────────────────────────────
print('[3/4] Training Random Forest (2–5 min on full data)...')
t0 = time.time()
rf_model = RandomForestRegressor(
    featuresCol='features', labelCol=LABEL_COL,
    predictionCol='prediction',
    numTrees=50, maxDepth=10,
    subsamplingRate=0.8, seed=42
).fit(train_df)
rf_preds = rf_model.transform(test_df)
results['Random Forest'] = {**evaluate(rf_preds),
    'time': round(time.time()-t0,1), 'preds': rf_preds,
    'importances': rf_model.featureImportances}
r = results['Random Forest']
print(f'RMSE={r["rmse"]:.3f}  MAE={r["mae"]:.3f}  R²={r["r2"]:.4f}  ({r["time"]}s)')

In [ ]:
# ── Model 4: Gradient Boosted Trees ───────────────────────
print('[4/4] Training GBT (3–7 min on full data)...')
t0 = time.time()
gbt_model = GBTRegressor(
    featuresCol='features', labelCol=LABEL_COL,
    predictionCol='prediction',
    maxIter=50, maxDepth=8,
    stepSize=0.1, subsamplingRate=0.8, seed=42
).fit(train_df)
gbt_preds = gbt_model.transform(test_df)
results['GBT'] = {**evaluate(gbt_preds),
    'time': round(time.time()-t0,1), 'preds': gbt_preds,
    'importances': gbt_model.featureImportances}
r = results['GBT']
print(f'RMSE={r["rmse"]:.3f}  MAE={r["mae"]:.3f}  R²={r["r2"]:.4f}  ({r["time"]}s)')

---
## Section 7 — Model Comparison & Evaluation

In [ ]:
# ── Ranked summary table ───────────────────────────────────
summary = pd.DataFrame([
    {'Model': n, 'RMSE': round(m['rmse'],3),
     'MAE':  round(m['mae'],3),
     'R²':   round(m['r2'],4),
     'Train Time (s)': m['time']}
    for n, m in results.items()
]).sort_values('RMSE').reset_index(drop=True)
summary.insert(0,'Rank', range(1, len(summary)+1))
BEST = summary.iloc[0]['Model']

print('═'*65)
print('      MODEL COMPARISON   (lower RMSE/MAE = better)')
print('═'*65)
print(summary.to_string(index=False))
print('═'*65)
print(f'\nBest model: {BEST}')

In [ ]:
# ── VIZ: 3-metric comparison ──────────────────────────────
models     = summary['Model'].tolist()
bar_colors = [PAL[2] if m == BEST else PAL[0] for m in models]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Model Comparison — All 4 Algorithms', fontsize=15, fontweight='bold')

for ax, metric, title in zip(
    axes,
    ['RMSE','MAE','R²'],
    ['RMSE  (lower = better)','MAE  (lower = better)','R²  (higher = better)']
):
    vals = summary[metric].tolist()
    bars = ax.bar(models, vals, color=bar_colors, edgecolor='white', width=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height() + max(vals)*0.012,
                f'{v:.3f}', ha='center', fontsize=10, fontweight='bold')
    ax.set_title(title); ax.tick_params(axis='x', rotation=12)
    bars[models.index(BEST)].set_edgecolor('#FFC107')
    bars[models.index(BEST)].set_linewidth(2.5)

plt.tight_layout(); save('viz_12_model_comparison'); plt.show()

In [ ]:
# ── VIZ: Actual vs Predicted + Residuals (best model) ─────
best_preds = results[BEST]['preds']
sp = (
    best_preds.select(LABEL_COL,'prediction')
    .sample(fraction=min(1.0, 15000/test_df.count()), seed=7)
    .toPandas().dropna()
)
sp['residual'] = sp[LABEL_COL] - sp['prediction']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle(f'🏆 Best Model — {BEST}', fontsize=14, fontweight='bold')

lim = max(abs(sp[LABEL_COL].quantile(0.99)), abs(sp['prediction'].quantile(0.99)))
axes[0].scatter(sp[LABEL_COL], sp['prediction'], alpha=0.25, s=8, color=PAL[2])
axes[0].plot([-lim,lim],[-lim,lim],'r--',lw=1.5,label='Perfect prediction')
axes[0].set_xlim(-lim,lim); axes[0].set_ylim(-lim,lim)
axes[0].set_xlabel('Actual ArrDelay (min)'); axes[0].set_ylabel('Predicted ArrDelay (min)')
axes[0].set_title('Actual vs Predicted'); axes[0].legend()

axes[1].hist(sp['residual'], bins=70, color=PAL[3], edgecolor='white', lw=0.4)
axes[1].axvline(0, color='red', lw=1.5, ls='--', label='Zero error')
axes[1].axvline(sp['residual'].mean(), color='gold', lw=1.5, ls='-.',
                label=f'Mean residual: {sp["residual"].mean():.2f}')
axes[1].set_xlabel('Residual (Actual − Predicted)'); axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution'); axes[1].legend()

plt.tight_layout(); save('viz_13_best_model_analysis'); plt.show()

In [ ]:
# ── VIZ: Feature importances (tree models) ────────────────
tree_res = {k: v for k, v in results.items() if 'importances' in v}
n = len(tree_res)

if n > 0:
    fig, axes = plt.subplots(1, n, figsize=(8*n, 5))
    if n == 1: axes = [axes]
    fig.suptitle('Feature Importances — Tree Models', fontsize=14, fontweight='bold')

    for ax, (name, m) in zip(axes, tree_res.items()):
        imp = pd.DataFrame({
            'feature':    FEATURE_COLS,
            'importance': m['importances'].toArray()
        }).sort_values('importance')
        c_imp = [PAL[0] if v < imp['importance'].median() else PAL[2]
                 for v in imp['importance']]
        ax.barh(imp['feature'], imp['importance'], color=c_imp, edgecolor='white')
        for i, (_, row) in enumerate(imp.iterrows()):
            ax.text(row['importance']+0.001, i, f'{row["importance"]:.3f}',
                    va='center', fontsize=9)
        ax.set_xlabel('Importance'); ax.set_title(name)

    plt.tight_layout(); save('viz_14_feature_importance'); plt.show()

---
## Section 8 — Conclusion

In [ ]:
best_rmse = results[BEST]['rmse']
best_r2   = results[BEST]['r2']

print('═'*65)
print('      US FLIGHT DELAY PREDICTOR — FINAL SUMMARY')
print('═'*65)
print(f'  Dataset : US BTS On-Time Performance — Full Range')
print(f'  Rows    : {TOTAL:,} flights after cleaning')
print(f'  Years   : {int(year_summary.Year.min())} → {int(year_summary.Year.max())}')
print(f'  Features: {FEATURE_COLS}')
print(f'  Target  : {LABEL_COL}')
print()
print('  Results (ranked by RMSE):')
for _, row in summary.iterrows():
    crown = '  ← WINNER' if row['Model'] == BEST else ''
    print(f"  {int(row['Rank'])}. {row['Model']:<22} "
          f"RMSE={row['RMSE']:.2f}  MAE={row['MAE']:.2f}  R²={row['R²']:.3f}{crown}")
print()
print(f'  Winner : {BEST}')
print(f'  RMSE   : {best_rmse:.2f} min  → predictions are off by ~{best_rmse:.0f} min on avg')
print(f'  R²     : {best_r2:.3f}  → explains {best_r2*100:.1f}% of variance in ArrDelay')
print('═'*65)

train_df.unpersist()
test_df.unpersist()
df.unpersist()
print('\nSpark caches released')